# 03 · Feature engineering

**Guideline covered:** Feature engineering

**What this notebook does.** Turns cleaned text into the inputs the models need.

- **Labels (cell F1):** star ratings become sentiment labels: 1–2 stars = negative, 3 = neutral, 4–5 = positive (`label3`), plus a two-class version without 3-star reviews (`label2`). A fixed **70 / 15 / 15 split** (train / validation / test; 59,605 / 12,772 / 12,773 reviews), stratified by app and label, with `random_state=42`.
- **TF-IDF features (cell F2):** word and word-pair weights, fitted on the training set only to avoid data leakage.
- **Hand-crafted features (cell F3):** VADER sentiment score, review length, exclamation marks, share of capital letters, money words, negation, emoji count. Correlation with the star rating is reported (negation −0.52, length −0.46, VADER +0.42).
- The "company replied" flag is created for business analysis only and is deliberately **not** used as a model input, because replies happen after the review (leakage).

Further features are built where they are used: sentence embeddings (notebook 05) and keyword theme indicators for the business models (notebook 07, cell M1).

**Inputs:** `clean_reviews.parquet` · **Outputs:** `model_data.parquet`, `tfidf_X.npz`, `tfidf_vectorizer.joblib`

**How to run:** open in Google Colab, keep the project folder at `MyDrive/upi-complaint-analytics` (or change `BASE` in the first code cell) and run the cells from top to bottom. The outputs shown are from the original run.

Feature engineering (Phase 5)

Cell F1: sentiment labels and the fixed data split

In [ ]:
import pandas as pd, numpy as np, re, emoji
from sklearn.model_selection import train_test_split
BASE = '/content/drive/MyDrive/upi-complaint-analytics'

df = pd.read_parquet(f'{BASE}/data/processed/clean_reviews.parquet')
data = df[~df.indic_only].copy().reset_index(drop=True)      # English-text reviews only

# Labels from star ratings: 1-2 = negative, 3 = neutral, 4-5 = positive
data['label3'] = pd.cut(data.rating, bins=[0, 2, 3, 5],
                        labels=['negative', 'neutral', 'positive']).astype(str)
data['label2'] = data.label3.where(data.label3 != 'neutral')   # empty for 3-star (binary version)

# 70 / 15 / 15 split, balanced by app AND label, fixed with random_state=42
strat = data.app + '_' + data.label3
train_idx, temp_idx = train_test_split(data.index, test_size=0.30, stratify=strat, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=strat[temp_idx], random_state=42)
data['split'] = 'train'
data.loc[val_idx, 'split'] = 'val'
data.loc[test_idx, 'split'] = 'test'

print(data.split.value_counts())
print('\nLabel share per split:\n', pd.crosstab(data.split, data.label3, normalize='index').round(3))
print('\nLabel share per app:\n', pd.crosstab(data.app, data.label3, normalize='index').round(3))

split
train    59605
test     12773
val      12772
Name: count, dtype: int64

Label share per split:
 label3  negative  neutral  positive
split                              
test       0.185    0.039     0.776
train      0.185    0.039     0.776
val        0.185    0.039     0.776

Label share per app:
 label3      negative  neutral  positive
app                                    
BHIM           0.536    0.086     0.377
Google Pay     0.218    0.043     0.739
Paytm          0.118    0.019     0.863
PhonePe        0.110    0.038     0.851


Cell F2: TF-IDF features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp, joblib

def emoji_tokens(e):
    return ' '.join('emj_' + re.sub(r'\W+', '_', emoji.demojize(x).strip(':')) for x in e) if e else ''

# text_full + emojis as single tokens + "ok" put back (these were the empty-text cases)
data['tfidf_text'] = (data.text_full + ' ' + data.emojis.fillna('').apply(emoji_tokens)
                      + np.where(data.text_light.str.contains(r'\bok\b'), ' ok', '')).str.strip()
print('Reviews still empty for TF-IDF:', (data.tfidf_text == '').sum())

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, max_features=20000, sublinear_tf=True)
tfidf.fit(data.loc[data.split == 'train', 'tfidf_text'])        # learn vocabulary from TRAIN only
X = tfidf.transform(data.tfidf_text)
print('TF-IDF matrix (reviews x features):', X.shape)

sp.save_npz(f'{BASE}/data/processed/tfidf_X.npz', X)
joblib.dump(tfidf, f'{BASE}/data/processed/tfidf_vectorizer.joblib')

tr = (data.split == 'train').values
vocab = np.array(tfidf.get_feature_names_out())
for lab in ['negative', 'positive']:
    m = np.asarray(X[tr & (data.label3 == lab).values].mean(axis=0)).ravel()
    print(f'\nTop terms in {lab} reviews:', ', '.join(vocab[m.argsort()[::-1][:20]]))

Reviews still empty for TF-IDF: 398
TF-IDF matrix (reviews x features): (85150, 6777)

Top terms in negative reviews: not, bad, good, work, not work, use, no, payment, open, otp, time, useless, please, slow, nice, money, show, issue, update, unable

Top terms in positive reviews: good, nice, emj_thumbs_up, super, ok, good emj_thumbs_up, well, excellent, nice emj_thumbs_up, payment, service, thank, great, fast, emj_smiling_face_with_smiling_eyes, easy, good service, experience, aap, use


Cell F3: VADER score, metadata features, and save

In [ ]:
!pip install -q vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
data['vader'] = data.text_light.apply(lambda t: sia.polarity_scores(t)['compound'])

orig = data.review_text
MONEY = r'\b(?:money|amount|rupees?|rs|inr|refund\w*|deduct\w*|debit\w*|credit\w*|balance|cashback)\b|₹'
data['n_words'] = data.words_no_emoji
data['n_exclaim'] = orig.str.count('!')
letters = orig.str.count(r'[A-Za-z]')
data['caps_ratio'] = (orig.str.count(r'[A-Z]') / letters.replace(0, np.nan)).fillna(0).round(3)
data['has_money'] = data.text_light.str.contains(MONEY, regex=True).astype(int)
data['has_negation'] = data.text_light.str.contains(r'\b(?:not|no|never|nothing)\b', regex=True).astype(int)
data['n_emojis'] = orig.apply(emoji.emoji_count)
data['has_reply'] = data.company_reply.notna().astype(int)   # for business analysis only, NOT a model input

feats = ['vader', 'n_words', 'n_exclaim', 'caps_ratio', 'has_money', 'has_negation', 'n_emojis', 'helpful_votes']
print('Spearman correlation with star rating:\n',
      data[feats + ['rating']].corr(method='spearman')['rating'].drop('rating').round(3).sort_values())
print('\nAverages by star rating:\n',
      data.groupby('rating')[['vader', 'n_words', 'has_money', 'has_negation', 'n_emojis']].mean().round(2))

data.to_parquet(f'{BASE}/data/processed/model_data.parquet', index=False)
print('\nSaved model_data.parquet:', data.shape)

Spearman correlation with star rating:
 has_negation    -0.521
n_words         -0.456
has_money       -0.248
helpful_votes   -0.235
caps_ratio      -0.210
n_exclaim       -0.062
n_emojis         0.085
vader            0.423
Name: rating, dtype: float64

Averages by star rating:
         vader  n_words  has_money  has_negation  n_emojis
rating                                                   
1       -0.13    19.00       0.17          0.54      0.25
2        0.08    16.54       0.12          0.43      0.14
3        0.24    11.54       0.08          0.26      0.18
4        0.39     6.03       0.04          0.09      0.23
5        0.43     2.99       0.02          0.03      0.30

Saved model_data.parquet: (85150, 28)
